<a href="https://colab.research.google.com/github/afirdousi/pytorch-basics/blob/main/007_computer_vision_cnn_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from torch import nn

import torchvision
from torchvision import datasets
from torchvision import transforms # (Read: https://pytorch.org/vision/stable/transforms.html)
from torchvision.transforms import ToTensor

import matplotlib.pyplot as plt

import pandas as pd
import numpy as np
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using PyTorch version = { torch.__version__ }")
print(f"Using device = { device }")  # We will be doing device agnostic code in this tutorial

Using PyTorch version = 2.0.1+cu118
Using device = cuda


In [ ]:
# Intro to Convolutional Neural Networks  (aka as CNN)
# CNNs are also known as ConvNets.
# CNNs are known for their capabilities to find patterns in visual data

# General Structure of a CNN (pretty much same as normal vision model)

# 1 >Input Image
# 2 > Input Layer
# 3 > Pre Processing
# 4 > Convolution Layer (nn.ConvXd()) - Extracts/learns most important features from target images
# 5 > Hidden Activation / Non Linear Activation (nn.ReLU()) - Adds non-linearity to learned features (non-straight lines)
# 6 > Pooling Layer (nn.MaxPool2d()) - Reduces dimentionality of learned image features
# 7 > Output layer/linear layer (nn.Linear) - Takes learned features and outputs them in shape of target labels
# 8 > Output activation (torch.sigmoid) - Converts output logits to prediction probablities

# 4,5,6 can be combined in many different ways and can be repeated many times as a set of conv > relu > pooling

# Deeper CNN (Current research area)
# Having multiple layers of conv > relu > pooling
# Current: The accepted idea is the more layers you add to your CNN, the more deeper pattern it will learn

In [ ]:
# Setup Training Data
train_data = datasets.FashionMNIST(
    root = "data", # which folder to download data to
    train = True, # all/most of built in datasets in PyTroch are already divided into train and testing datasets
    download = True, # do want to download?
    transform = torchvision.transforms.ToTensor(), # how do we want to transform the data
    target_transform = None #. how do we want to transform the labels/targets?
)

# Setup Test Data
test_data = datasets.FashionMNIST(
    root = "data",
    train = False,
    download = True,
    transform = torchvision.transforms.ToTensor(),
    target_transform= None
)

In [ ]:
# Learn what happens inside CNN on CNN Explainer: https://poloclub.github.io/cnn-explainer/
# Check: https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html
# Learn about hyperparameters of CNN on https://poloclub.github.io/cnn-explainer/ (search "Understanding Hyperparameters")

# Let's code
class FashionMNISTModelV2(nn.Module):
  """
  This model architecture will replicate the TinyVGG
  model from CNN explainer website
  """
  # Learn about Orginal VGG: https://medium.com/analytics-vidhya/vggnet-architecture-explained-e5c7318aa5b6

  def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
    super().__init__()
    # In CNN, things are generally referred to as Convolutional Block, think of a convolutional block as a set of conv > relu > pooling
    # deeper CNNs will have multiple conv blocks
    self.conv_block_1 = nn.Sequential(
        nn.Conv2d( in_channels = input_shape, # Conv2d because we are working on a 2D image
                  out_channels = hidden_units,
                   kernel_size = 3, # this could be 3x3 or 3
                   stride = 1,
                   padding = 1), # these are all hyperparametrs used for conv 2d
        nn.ReLU(),
        nn.Conv2d( in_channels = hidden_units,
                  out_channels = hidden_units,
                   kernel_size = 3,
                   stride = 1,
                   padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2)  # takes the max value form a window of nxn witin an image
        # In general, one particular theme in deep neural networks is the images
        # will continue go down smaller and smaller the deeper you go in the network

    )

    self.conv_block_2 = nn.Sequential(
        nn.Conv2d( in_channels = hidden_units,
                  out_channels = hidden_units,
                   kernel_size = 3,
                   stride = 1,
                   padding = 1),
        nn.ReLU(),
        nn.Conv2d( in_channels = hidden_units,
                  out_channels = hidden_units,
                   kernel_size = 3,
                   stride = 1,
                   padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2)
    )

    # finally we will have a classifier layer (where we flatten and classify into target labels)

    # compare this layer to previous simpler FashionMNIST models we made
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features = hidden_units * 0, # there is a trick to calculate this number which we have marked as 0 (more on it later)
                  out_features= output_shape)
    )

    def forward(self, x):
      x = self.conv.conv_block_1(x)
      print(x.shape)
      x = self.conv.conv_block_2(x)
      print(x.shape)
      x = self.conv.classifier(x)
      print(x.shape)

In [ ]:
torch.manual_seed(42)

model_1 = FashionMNISTModelV2(input_shape= 1, # since we have black and white images (Vs CNN explainer guys have color videos)
                              hidden_units = 10,
                              output_shape = len(train_data.classes)).to(device)

/usr/local/lib/python3.10/dist-packages/torch/nn/init.py:405: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")
